<a href="https://colab.research.google.com/github/qaz9391/LINE-BOT-B12090026/blob/main/0501_Colab_ReplyBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
google_gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://blazer-broadly-harpist.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://blazer-broadly-harpist.ngrok-free.dev


True

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    print("EVENT: ", event)
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=event.message.text),#因為這邊程式碼傳兩次#
                            TextMessage(text=event.message.text)]#因為這邊程式碼傳兩次#
            )
        )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616768251208400966","quoteToken":"VKHg0dgKBFxTZ62f-CINDOAS1CgiPKEckO890yphmVFqsS6_GvkEKwLEOD2ieIGicjfa_1Tu_J7FwkdIyku8rqvIS7WvseSwoRBWGiKlzwfRi8tT0bMkAJs4SMt5nTwIejix0um_atUheVE2AeVajw","markAsReadToken":"iqqGXOBIVdVA3UQYqAwKRcyh0pPV5mSWKcIarw8tIIw85Xmb8ITzZdFgwsYrKhkAExTj3eJhSXfzn2qCRiLP_wm-baQTw3_rg8jYytEZQPKxvQRD6DUFCE3kEH26sJpIGubPW5LKIC4AfUrIWybDK-bFk41qjDqjc3qlwMIGCeGWRqBfvXh6eIR2x5zaoe2rEAVZCtOQcYBl92rkGk8XZw","text":"嗨"},"webhookEventId":"01KT5N2TWNJV0KRAJH821MFHHF","deliveryContext":{"isRedelivery":false},"timestamp":1780453829276,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"acbb2db886d6487599ad87377492db83","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='U4a7effcc683b4b1daf00f53459845948') timestamp=1780453829276 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KT5N2TWNJV0KRAJH821MFHH

INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 02:30:30] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616768261928780330","quoteToken":"CXSYwACk29iPanzWCPSb67l4ZqN5vIpweH5PxKpR-H_FzIeQz6rHxu5Y40HOyjx1Icl9nJkWXvVMsMNOdSJu4Bcakl2T6xsOB_6qDtRp54CmZ6DLKoYlJTHw1UBG_fwbz89SGQRWvZLXheMkSV3DzQ","markAsReadToken":"NnrR5aN9nJOdBJCIb1L8W476i_5zKvNayadPpwsTXSMC26MWeYVC5CnuKtGlk6Q3ttqrTxKT3TTLBdeOzLzeWyU-XN957vq1TSrtAzFqGK7THGdv6iNEJminPXfytNVjUqn4SiSfnOhWnuqROt2lnfrcjty2H_kLRr_9q0eEeAeMi8Wj2QmXigSmFxv_VEtoqO1LBg-RXvPPl9YMbRKSVg","text":"我是謝嘉哲"},"webhookEventId":"01KT5N312M1TTCTH9F0ZMFT89C","deliveryContext":{"isRedelivery":false},"timestamp":1780453835695,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"31d87c4cecec40a9b8a8fff4f2f954ad","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='U4a7effcc683b4b1daf00f53459845948') timestamp=1780453835695 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KT5N312M1TTCTH9F0ZM

INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 02:30:36] "POST / HTTP/1.1" 200 -
